# Phishing Email Detection – Dataset Preparation (Merge + Clean)

This notebook prepares the email datasets for phishing detection.

**Datasets used**
- **Enron**: mostly legitimate emails (0) + some phishing/spam (1)
- **Nazario**: phishing emails (1)

**Output**
- A single cleaned dataset saved to: `../data/processed/clean_emails.csv`

In [59]:
import pandas as pd
import re
from pathlib import Path

## 1) Load Raw Datasets

We load the raw CSV files, then create one unified `text` field by combining **subject + body**.

In [60]:

# Load datasets
enron = pd.read_csv("../data/raw/enron.csv")
nazario = pd.read_csv("../data/raw/nazario.csv")

# Create unified text column
enron["text"] = enron["subject"].fillna("") + " " + enron["body"].fillna("")
nazario["text"] = nazario["subject"].fillna("") + " " + nazario["body"].fillna("")

# Keep only necessary columns
enron = enron[["text", "label"]]
nazario = nazario[["text", "label"]]

# Add source labels
enron["source"] = "enron"
nazario["source"] = "nazario"

print("ENRON shape:", enron.shape)
print("NAZARIO shape:", nazario.shape)

ENRON shape: (29767, 3)
NAZARIO shape: (1565, 3)


## 2) Check Label Distribution (Before Merging)

This helps confirm whether the datasets are balanced or imbalanced.

In [61]:
print("ENRON label counts:\n", enron["label"].value_counts())
print("\nNAZARIO label counts:\n", nazario["label"].value_counts())

ENRON label counts:
 label
0    15791
1    13976
Name: count, dtype: int64

NAZARIO label counts:
 label
1    1565
Name: count, dtype: int64


## 3) Merge Datasets

We merge both datasets into one combined dataset before cleaning and feature extraction.

In [62]:
merged = pd.concat([enron, nazario], ignore_index=True)
print("Merged shape:", merged.shape)
print("\nMerged label counts:\n", merged["label"].value_counts())
merged.head()

Merged shape: (31332, 3)

Merged label counts:
 label
0    15791
1    15541
Name: count, dtype: int64


,text,label,source
0,"hpl nom for may 25 , 2001 ( see attached file ...",0,enron
1,re : nom / actual vols for 24 th - - - - - - -...,0,enron
2,"enron actuals for march 30 - april 1 , 201 est...",0,enron
3,"hpl nom for may 30 , 2001 ( see attached file ...",0,enron
4,"hpl nom for june 1 , 2001 ( see attached file ...",0,enron


## 4) Clean Text

We create a `clean_text` column:
- lowercase
- remove URLs
- remove punctuation and numbers
- remove extra whitespace

This helps TF-IDF and models learn better.

In [63]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)   # remove urls
    text = re.sub(r"[^a-z\s]", " ", text)  # remove punctuation & numbers
    text = re.sub(r"\s+", " ", text)       # remove extra spaces
    return text.strip()

merged["clean_text"] = merged["text"].apply(clean_text)

merged[["text", "clean_text"]].head()

,text,clean_text
0,"hpl nom for may 25 , 2001 ( see attached file ...",hpl nom for may see attached file hplno xls hp...
1,re : nom / actual vols for 24 th - - - - - - -...,re nom actual vols for th forwarded by sabrae ...
2,"enron actuals for march 30 - april 1 , 201 est...",enron actuals for march april estimated actual...
3,"hpl nom for may 30 , 2001 ( see attached file ...",hpl nom for may see attached file hplno xls hp...
4,"hpl nom for june 1 , 2001 ( see attached file ...",hpl nom for june see attached file hplno xls h...


## 5) Remove Duplicates

We remove duplicate emails based on the cleaned text to avoid training on repeated samples.

In [64]:
before = len(merged)
merged = merged.drop_duplicates(subset="clean_text").reset_index(drop=True)
after = len(merged)

print("Removed duplicates:", before - after)
print("Final dataset size:", after)
print("\nFinal label counts:\n", merged["label"].value_counts())

Removed duplicates: 1499
Final dataset size: 29833

Final label counts:
 label
1    15447
0    14386
Name: count, dtype: int64


## 6) Save Final Clean Dataset (Only Once)

This is the dataset used for all baseline models.

Saved to: `../data/processed/clean_emails.csv`

In [66]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = PROCESSED_DIR / "clean_emails.csv"
merged.to_csv(OUT_PATH, index=False)

print("✅ Saved:", OUT_PATH)

✅ Saved: ..\data\processed\clean_emails.csv
